# 07. Named Entity Recognition — Azure AI Language (`recognize_entities`, prebuilt NER)

This notebook calls the **dedicated Azure AI Language** prebuilt Named Entity Recognition model via
`TextAnalyticsClient.recognize_entities()` — the direct counterpart to the LLM-prompted extraction in
`01_named_entitities.py`. It runs the same style of support-ticket text through Language's built-in NER model
and prints each entity's category, optional subcategory, and confidence score.

Compare this notebook's output structure and category set directly against notebook 01: the prebuilt model
uses a *fixed*, Microsoft-defined category taxonomy (`Person`, `Organization`, `DateTime`, `Quantity`, etc.) —
it has no concept of domain-specific categories like `ticket_id` or `sla_tier` that the LLM prompt in notebook
01 was free to invent.

**Difficulty:** Beginner


## Prerequisites

**Python packages:**
- `azure-ai-textanalytics` — **not yet in the repo's root `requirements.txt`**, install with:
  ```
  pip install azure-ai-textanalytics
  ```
- `python-dotenv` (already in `requirements.txt`)

**Azure resources required:**
- An Azure AI Language resource with a key and endpoint

**Environment variables** (add to root `.env`):
```
AZURE_LANGUAGE_ENDPOINT=https://<your-language-resource>.services.ai.azure.com/
AZURE_LANGUAGE_KEY=<your-language-resource-key>
```


## What You'll Learn

- How to call `recognize_entities()` for prebuilt Named Entity Recognition
- The response shape: `.category`, `.subcategory`, `.text`, `.confidence_score`
- How prebuilt NER's fixed category taxonomy compares to Custom NER (train-your-own) and to LLM-prompted
  extraction (notebook 01)
- When each of the three approaches (prebuilt, custom, LLM prompt) is the right engineering choice


### Step 1 — Create the Text Analytics client

Same `TextAnalyticsClient` + `AzureKeyCredential` pattern as notebooks 05 and 06 — NER is just another
operation on the same client.

💡 **Exam tip:** AI-102's Language service objectives explicitly distinguish **prebuilt NER** (zero training,
fixed general-purpose categories, works out of the box) from **Custom NER** (you label example documents in
Language Studio / with the SDK, train a model, and get back your own domain-specific categories). This
notebook exercises the prebuilt path.

🔄 **Alternatives:** N/A for client setup — see notebook 05 for the REST-vs-SDK alternative.


In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

load_dotenv()

endpoint = os.environ["AZURE_LANGUAGE_ENDPOINT"]
api_key = os.environ["AZURE_LANGUAGE_KEY"]

client = TextAnalyticsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
)

### Step 2 — Run prebuilt NER over the support ticket text

Same ticket text used in notebooks 01 and 02, run through `recognize_entities(documents, language="en")`
instead of an LLM prompt. This lets you directly compare which entities each approach surfaces.

💡 **Exam tip:** Azure AI Language's prebuilt NER model recognizes general categories such as `Person`,
`Location`, `Organization`, `DateTime`, `Quantity`, `Event`, `Product`, `Skill`, `URL`, `Email`, and
`PhoneNumber` (exact list can evolve with service updates) — notice this is a *fixed* taxonomy defined by
Microsoft, not something you configure per call. Domain-specific concepts like "ticket ID" or "SLA tier" (from
notebook 01/02) simply have no home in this taxonomy without Custom NER training.

🔄 **Alternatives:** If your entities of interest fall outside the prebuilt taxonomy, your two options are (a)
**Custom NER** — label training examples and train a model in Language Studio, or (b) LLM prompting (notebook
01) — zero training data needed but no confidence scores and less deterministic behavior.


In [ ]:
documents = [
    "Hi, this is Sarah Chen from Acme Logistics writing in again about "
    "ticket TKT-1042. Our Gold-tier SLA promises a 4 hour response time, "
    "and we are now at hour 6 with no update. The VPN client keeps "
    "dropping every 10 minutes on our Windows fleet since the rollout of "
    "CloudXeus Connect v3.2 last Tuesday. If this isn't resolved by end "
    "of day Friday we will be requesting the $500 SLA breach credit "
    "outlined in our contract.",
]

response = client.recognize_entities(documents, language="en")
results = [doc for doc in response if not doc.is_error]

### Step 3 — Print each entity, its category/subcategory, and confidence

Each `CategorizedEntity` has `.category` (the top-level type, e.g. `"Person"`), an optional `.subcategory`
(a finer-grained tag some categories provide, e.g. `"Age"` under `Quantity`, or `None` when not applicable),
`.text` (the exact matched substring), and `.confidence_score`.

💡 **Exam tip:** `subcategory` is `None`/empty for many entities and populated for others — code defensively
around it (as this script does with the conditional `f" / {entity.subcategory}"` only when present) rather
than assuming every entity has one. This detail — handling optional fields in the SDK response models — is the
kind of thing AI-102 code-reading questions probe.

🔄 **Alternatives:** Run the exact same `documents` list through `01_named_entitities.py`'s LLM-prompt approach
and compare: does the prebuilt model catch `"$500"` as a monetary amount / quantity? Does it catch
`"TKT-1042"` at all (it likely won't, since ticket IDs aren't a prebuilt category)? This comparison is a good
way to internalize when prebuilt NER falls short of what a well-prompted LLM can do — and vice versa (prebuilt
NER never hallucinates an entity that isn't in the text).


In [ ]:
for idx, doc in enumerate(results):
    print(f"--- Document {idx + 1}: Prebuilt NER Results ---")
    for entity in doc.entities:
        subcat = f" / {entity.subcategory}" if entity.subcategory else ""
        print(f"  [{entity.category}{subcat}] '{entity.text}'  (confidence: {entity.confidence_score:.2f})")
    print()

## Summary

This notebook ran Azure AI Language's prebuilt Named Entity Recognition model over the same support ticket
text used in notebook 01, printing each entity's fixed-taxonomy category, optional subcategory, and confidence
score. Where notebook 01's LLM prompt could freely invent categories like `ticket_id` and `sla_tier`, this
prebuilt model is limited to Microsoft's general-purpose category set — but in exchange gives you calibrated
confidence scores and fully deterministic, auditable behavior with no prompt to maintain.


## Try It Yourself

1. **Easy:** Run this notebook's `documents` text through `01_named_entitities.py` as well, and list which
   entities each approach found that the other missed.
2. **Intermediate:** Filter the printed entities down to only those with `confidence_score >= 0.7`, to see how
   much (if any) low-confidence noise the prebuilt model produces on this text.
3. **Advanced:** Look up how to train a **Custom NER** project in Language Studio for the `ticket_id` and
   `sla_tier` categories this text contains, and sketch (in a markdown cell) what a minimal labeled training
   set for those two categories would need to look like.
